# S03 · Laboratorio: manipulación y validación de datos de Olist
### Actividades en Python con pandas y NumPy

**Diplomado:** Métodos estadísticos y matemáticos para la ciencia de datos · Facultad de Ciencias, UNAM  
**Sesión 03 · 30 de octubre de 2026**

**Pregunta de trabajo:** *¿Qué categorías concentran la mayor suma de precios de artículos en pedidos entregados y qué verificaciones necesitamos para confiar en ese resultado?*

Trabajarás con los datos reales publicados en [Olist / GitHub](https://github.com/olist/work-at-olist-data). El laboratorio tiene tres niveles: **aplicación esencial**, **interpretación y validación**, y **extensión opcional**. Puedes trabajar individualmente o en equipo y utilizar la demostración como referencia.

**Resultados esperados:** tablas calculadas con código, comprobaciones de integridad y respuestas breves que justifiquen las decisiones. No es una entrega evaluada durante la sesión.

> Los importes de la tabla de artículos son **precios registrados por fila**. No deben interpretarse directamente como ingresos netos o pagos recibidos.

## 0. Preparar el entorno y los archivos

Clona el repositorio desde la terminal, si todavía no lo tienes:

```bash
git clone https://github.com/olist/work-at-olist-data.git
```

Guarda este notebook en una carpeta hermana de `work-at-olist-data` y selecciona el kernel de Python en VS Code o Jupyter. Instala los paquetes necesarios en el entorno elegido:

```bash
python -m pip install pandas numpy ipykernel
```

También puedes instalar las dependencias desde el archivo `requirements.txt` del paquete del curso. Conserva intactos los CSV originales. Si los datos están en otra ubicación, utiliza la variable de entorno `OLIST_DATA_DIR` para indicar la carpeta `datasets`.

## 1. Preparación: importa y explora los datos

Estas celdas preparan las cuatro tablas necesarias. Ejecútalas antes de resolver las actividades. Observa las diferencias entre las unidades de análisis: un pedido, una fila de artículo y un producto.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

In [ ]:
REQUIRED_FILES = (
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_products_dataset.csv",
    "product_category_name_translation.csv",
)


def localizar_datos() -> Path:
    """Devuelve la carpeta datasets/ de Olist o indica cómo prepararla."""
    candidatos = []
    definida = os.environ.get("OLIST_DATA_DIR")
    if definida:
        candidatos.append(Path(definida).expanduser())

    ubicacion = Path.cwd().resolve()
    for base in (ubicacion, *ubicacion.parents):
        candidatos.extend([
            base / "work-at-olist-data" / "datasets",
            base / "datasets",
            base.parent / "work-at-olist-data" / "datasets",
        ])

    for ruta in candidatos:
        if ruta.is_dir() and all((ruta / nombre).is_file() for nombre in REQUIRED_FILES):
            return ruta
    raise FileNotFoundError(
        "No encontré los CSV de Olist. Ejecuta 'git clone "
        "https://github.com/olist/work-at-olist-data.git' en la "
        "carpeta del curso y coloca este notebook en una carpeta hermana "
        "del repositorio; o establece OLIST_DATA_DIR con la ruta a datasets/."
    )


DATA_DIR = localizar_datos()
print("Datos encontrados en:", DATA_DIR)

In [ ]:
orders = pd.read_csv(
    DATA_DIR / "olist_orders_dataset.csv",
    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"],
)
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
categories = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

for nombre, tabla in {
    "orders": orders,
    "items": items,
    "products": products,
    "categories": categories,
}.items():
    print(f"{nombre:12s}: {len(tabla):>8,} filas × {tabla.shape[1]:>2} columnas")

display(orders.head(3))
display(items.head(3))
display(products.head(3))

### Reconocimiento del modelo relacional

Antes de seguir, responde brevemente:

- ¿Cuál es la llave que identifica un pedido en `orders`?
- ¿Por qué `order_id` no es único en `items`?
- ¿Qué relación esperas entre `items` y `products` al utilizar `product_id`?

## 2. Datos iniciales para las actividades

La siguiente celda incorpora el estado de los pedidos a la tabla de artículos. `validate="many_to_one"` indica la cardinalidad esperada y permite detectar llaves repetidas en la tabla derecha.

A partir de `articulos_entregados`, **tú construirás la unión con los productos y sus categorías** en el nivel esencial.

In [ ]:
# Base preparada: una fila por artículo con información del pedido.
assert orders["order_id"].is_unique, "Revisa las llaves de orders."

articulos_con_pedido = items.merge(
    orders[["order_id", "order_status", "order_purchase_timestamp"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Correspondencias de pedidos:")
display(articulos_con_pedido["_merge"].value_counts(dropna=False))

articulos_entregados = articulos_con_pedido.loc[
    articulos_con_pedido["order_status"].eq("delivered")
].drop(columns="_merge").copy()

print("Filas originales de artículos:", len(items))
print("Filas de artículos en pedidos entregados:", len(articulos_entregados))
display(articulos_entregados.head(3))

## A. Aplicación esencial · 25 minutos

**A1.** Une `articulos_entregados` con el catálogo `products` mediante `product_id`. Después incorpora la traducción de categorías mediante `product_category_name`.

**A2.** Crea la columna `categoria`, utilizando primero `product_category_name_english` y, cuando falte, el nombre original. Si ambos faltan, escribe `SIN_CATEGORIA`.

**A3.** Calcula por categoría: número de filas de artículos, número de pedidos distintos, suma de precios y mediana del precio. Ordena de mayor a menor por la suma.

**A4.** Identifica tres categorías con mayor suma y explica por qué la categoría con mayor suma no necesariamente tiene el precio mediano más alto.

Puedes consultar `merge`, `fillna`, `groupby`, `agg`, `sort_values` y `nunique` en la demostración. Utiliza `validate="many_to_one"` en las uniones si esa es la relación que verificaste.

In [ ]:
# A1. Une los artículos entregados con los catálogos.
# Guarda el resultado en una variable llamada analisis.

# analisis = ...

# A2. Completa la columna de categoría.
# analisis["categoria"] = ...

# A3. Construye una tabla llamada resumen_categorias.
# resumen_categorias = ...

# A4. Muestra y describe las primeras tres categorías.
# display(resumen_categorias.head(3))

**Respuesta A4.** Escribe aquí tu explicación y distingue entre *suma de precios* y *mediana de precios por fila*.

> Tu respuesta:

## B. Interpretación y validación · 25 minutos

**B1.** Comprueba la unicidad de `product_id` en `products` y de `product_category_name` en `categories`. ¿Qué consecuencias tendría encontrar duplicados antes de una unión muchos-a-uno?

**B2.** Si completaste A1, verifica que las uniones no hayan cambiado inesperadamente el número de filas respecto de `articulos_entregados`.

**B3.** Cuenta los artículos con categoría `SIN_CATEGORIA`. Explica por qué un valor faltante no debe borrarse automáticamente.

**B4.** Compara el número de filas del análisis con el número de pedidos distintos. ¿Qué representa cada cifra?

**B5.** Verifica que la suma global de `price` en `analisis` coincida con la suma por grupos en `resumen_categorias`. Si no coincide, investiga qué operación podría explicarlo.

Si todavía no has completado A1–A3, dedica este tiempo a terminar el nivel esencial.

In [ ]:
# B1. Unicidad de llaves:
# duplicados_productos = ...
# duplicados_categorias = ...

# B2. Invariantes de filas:
# print(...)

# B3 y B4. Faltantes y unidades de observación:
# print(...)

# B5. Comprueba totales con np.isclose(...):
# print(...)

**Interpretación B.** Anota tus hallazgos. Si una comprobación falla, explica cómo investigarías la causa antes de modificar los datos.

> Tu respuesta:

## C. Extensión computacional opcional · 20 minutos

**C1.** Escribe una función `top_categorias(tabla, n)` que regrese las `n` categorías con mayor suma de precios registrados de artículos.

**C2.** Comprueba que `n` sea un entero positivo y explica qué hacer si `n` excede el número de categorías disponibles.

**C3.** Utiliza un condicional o un ciclo en una tarea razonable (por ejemplo, informar el número de registros que corresponden a distintos estados). Compáralo con la operación vectorizada equivalente.

**C4.** Si dispones de un entorno de R, implementa una de las agrupaciones anteriores con `dplyr` y compara los resultados de ambos lenguajes.

Este apartado es voluntario. Si lo necesitas, utiliza estos minutos para terminar los niveles A y B.

In [ ]:
# C1–C2. Define una función con parámetros y validación.
# def top_categorias(tabla: pd.DataFrame, n: int = 5) -> pd.DataFrame:
#     ...

# C3. Compara una solución con ciclo y otra vectorizada.

# C4. Registra cualquier diferencia observada entre Python y R.

## 3. Cierre: ¿podemos confiar en el resultado?

Responde en cuatro o cinco frases:

1. ¿Cuál es la unidad de observación del resultado final?
2. ¿Qué se perdería si confundiéramos filas de artículos con pedidos?
3. ¿Qué efecto puede tener filtrar únicamente pedidos `delivered`?
4. ¿Qué información adicional necesitaríamos para hablar de ingresos efectivamente cobrados?
5. ¿Qué archivos y decisiones debe conocer otra persona para reproducir el análisis?

**Registro de procedencia:** desde una terminal situada en el repositorio de Olist, ejecuta `git rev-parse HEAD` y conserva el identificador del commit. Esto permite documentar la versión de los datos. No es necesario modificar ni publicar cambios en el repositorio original.

## Referencias

- Olist. [Work at Olist Data](https://github.com/olist/work-at-olist-data) (datos reales publicados en GitHub).
- pandas. [User Guide](https://pandas.pydata.org/docs/user_guide/).
- NumPy. [Documentación](https://numpy.org/doc/stable/).
- Chacon, S., y Straub, B. [Pro Git](https://git-scm.com/book/es/v2).

Los resultados dependen del estado de los archivos clonados. Las comprobaciones propuestas ayudan a detectar problemas de integridad, pero no sustituyen la evaluación del contexto, alcance y calidad de los datos.